# License Plate Recognition — Inference Pipeline (PP-OCRv5 vs PP-OCRv4)

Downloads the fine-tuned recognition models published by `PPOCR_License_Plate_Training.ipynb`
as a GitHub Release, runs the full PaddleOCR pipeline (detection + recognition) on the 30 real
photos that were held out of training, draws annotated bounding boxes/text/confidence, compares
PP-OCRv5 vs PP-OCRv4, and reports batch statistics and error analysis.

**Run the training notebook first** — it publishes the release (tag `trained-models`) this notebook
downloads from. Nothing to upload here, no Google Drive.

## 1. Setup

In [ ]:
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu121/
!pip install -q paddleocr pandas

In [ ]:
%cd /content

RELEASE_BASE = "https://github.com/parushurambannela/license-plate-recognition-paddleocr/releases/download/trained-models"
!wget -q {RELEASE_BASE}/v5_inference.zip -O v5_inference.zip
!wget -q {RELEASE_BASE}/v4_inference.zip -O v4_inference.zip
!wget -q {RELEASE_BASE}/test_images.zip -O test_images.zip

!unzip -oq v5_inference.zip -d v5_inference
!unzip -oq v4_inference.zip -d v4_inference
!unzip -oq test_images.zip -d test_images
print("Downloaded and unpacked release assets.")

## 2. Load Models

Text detection is left at its pipeline default for both, so the fine-tuned recognition model is
the only variable being compared.

In [ ]:
from paddleocr import PaddleOCR

common_kwargs = dict(use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False)

ocr_v5 = PaddleOCR(text_recognition_model_dir="/content/v5_inference", **common_kwargs)
ocr_v4 = PaddleOCR(text_recognition_model_dir="/content/v4_inference", **common_kwargs)
print("Both pipelines loaded.")

## 3. Test Images

30 real car photos set aside by the training notebook — never used for fine-tuning or
validation.

In [ ]:
import csv
import os

TEST_DIR = "/content/test_images"
with open(os.path.join(TEST_DIR, "labels.csv")) as f:
    test_samples = [{"path": os.path.join(TEST_DIR, row["image"]), "text": row["text"]} for row in csv.DictReader(f)]

print(f"Loaded {len(test_samples)} held-out test photos")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, sample in zip(axes.flat, test_samples[:8]):
    ax.imshow(Image.open(sample["path"]))
    ax.set_title(sample["text"], fontsize=9)
    ax.axis("off")
plt.suptitle("Sample held-out test photos (ground truth in title)")
plt.tight_layout()
plt.show()

## 4. Run Inference (Both Models)

In [ ]:
import time

def run_pipeline(ocr, samples):
    results = []
    for s in samples:
        t0 = time.time()
        res = ocr.predict(s["path"])[0].json["res"]
        results.append({
            "path": s["path"],
            "gt_text": s["text"],
            "rec_texts": res.get("rec_texts", []),
            "rec_scores": res.get("rec_scores", []),
            "rec_boxes": res.get("rec_boxes", []),
            "latency": time.time() - t0,
        })
    return results

v5_results = run_pipeline(ocr_v5, test_samples)
v4_results = run_pipeline(ocr_v4, test_samples)
print("Inference complete for both models.")

In [ ]:
def best_detection(entry):
    # A photo can contain other text (stickers, signage); keep only the highest-confidence box.
    if not entry["rec_texts"]:
        return None
    idx = max(range(len(entry["rec_scores"])), key=lambda i: entry["rec_scores"][i])
    return {"text": entry["rec_texts"][idx], "score": entry["rec_scores"][idx], "box": entry["rec_boxes"][idx]}

## 5. Annotated Results

In [ ]:
from PIL import ImageDraw

def draw_annotations(image_path, entry, color=(0, 200, 0)):
    img = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(img)
    for text, score, box in zip(entry["rec_texts"], entry["rec_scores"], entry["rec_boxes"]):
        x0, y0, x1, y1 = [int(v) for v in box]
        draw.rectangle([x0, y0, x1, y1], outline=color, width=3)
        label = f"{text} ({score:.2f})"
        draw.rectangle([x0, max(0, y0 - 18), x0 + 7 * len(label), y0], fill=color)
        draw.text((x0 + 3, max(0, y0 - 17)), label, fill=(255, 255, 255))
    return img

N_SHOW = 4
fig, axes = plt.subplots(N_SHOW, 3, figsize=(15, 4 * N_SHOW))
for row, (s, r5, r4) in enumerate(zip(test_samples[:N_SHOW], v5_results[:N_SHOW], v4_results[:N_SHOW])):
    axes[row, 0].imshow(Image.open(s["path"]))
    axes[row, 0].set_title(f"Original (GT: {s['text']})", fontsize=9)
    axes[row, 1].imshow(draw_annotations(s["path"], r5, color=(0, 160, 0)))
    axes[row, 1].set_title("PP-OCRv5", fontsize=9)
    axes[row, 2].imshow(draw_annotations(s["path"], r4, color=(200, 60, 0)))
    axes[row, 2].set_title("PP-OCRv4", fontsize=9)
    for c in range(3):
        axes[row, c].axis("off")
plt.tight_layout()
plt.show()

## 6. Side-by-Side Comparison

In [ ]:
import pandas as pd

rows = []
for s, r5, r4 in zip(test_samples, v5_results, v4_results):
    b5, b4 = best_detection(r5), best_detection(r4)
    rows.append({
        "image": os.path.basename(s["path"]),
        "ground_truth": s["text"],
        "v5_pred": b5["text"] if b5 else None,
        "v5_conf": round(b5["score"], 3) if b5 else None,
        "v5_match": bool(b5) and b5["text"] == s["text"],
        "v4_pred": b4["text"] if b4 else None,
        "v4_conf": round(b4["score"], 3) if b4 else None,
        "v4_match": bool(b4) and b4["text"] == s["text"],
    })

comparison_df = pd.DataFrame(rows)
comparison_df

## 7. Batch Processing & Summary Statistics

In [ ]:
import difflib

def char_accuracy(pred, gt):
    # Similarity ratio between predicted and ground-truth text (1.0 = identical).
    if not gt:
        return 0.0
    return difflib.SequenceMatcher(None, pred or "", gt).ratio()

comparison_df["v5_char_acc"] = comparison_df.apply(lambda r: char_accuracy(r["v5_pred"], r["ground_truth"]), axis=1)
comparison_df["v4_char_acc"] = comparison_df.apply(lambda r: char_accuracy(r["v4_pred"], r["ground_truth"]), axis=1)

summary_stats = pd.DataFrame({
    "model": ["PP-OCRv5", "PP-OCRv4"],
    "exact_match_accuracy": [comparison_df["v5_match"].mean(), comparison_df["v4_match"].mean()],
    "avg_char_accuracy": [comparison_df["v5_char_acc"].mean(), comparison_df["v4_char_acc"].mean()],
    "avg_confidence": [comparison_df["v5_conf"].mean(skipna=True), comparison_df["v4_conf"].mean(skipna=True)],
    "avg_latency_sec": [
        pd.Series([r["latency"] for r in v5_results]).mean(),
        pd.Series([r["latency"] for r in v4_results]).mean(),
    ],
})
display(summary_stats)

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for i, metric in enumerate(["exact_match_accuracy", "avg_char_accuracy", "avg_latency_sec"]):
    ax[i].bar(summary_stats["model"], summary_stats[metric], color=["tab:blue", "tab:orange"])
    ax[i].set_title(metric)
plt.tight_layout()
plt.show()

## 8. Error Analysis

In [ ]:
errors = comparison_df[~comparison_df["v5_match"] | ~comparison_df["v4_match"]]

if len(errors):
    display(errors[["image", "ground_truth", "v5_pred", "v4_pred"]])
else:
    print("Both models got every held-out photo exactly right.")

## 9. Export Results

In [ ]:
OUT_DIR = "/content/inference_results"
os.makedirs(OUT_DIR, exist_ok=True)

comparison_df.to_csv(os.path.join(OUT_DIR, "comparison.csv"), index=False)
summary_stats.to_csv(os.path.join(OUT_DIR, "summary_stats.csv"), index=False)
print("Results saved to", OUT_DIR, "(download via the Colab Files pane if you want to keep them)")